# Paper PoRT Official Artifact Probe

This notebook audits whether the official PoRT artifacts needed for `PORT_ARTIFACT_MODE=official` are publicly available or supplied through environment variables. It does not run model inference. Its job is to decide whether the next PoRT run can be paper-faithful, or whether we must recreate artifacts from public code/data and label them as recreated.

Required official artifacts:
- T5 AST/prefix compiler model.
- Post-judgment classifier base model.
- Post-judgment classifier head checkpoint.


In [1]:
from pathlib import Path
import hashlib
import importlib
import json
import os
import re
import subprocess
import sys
import time
import urllib.error
import urllib.parse
import urllib.request
import zipfile
from io import BytesIO

REPO_URL = 'https://github.com/toanthangO20/PoRT_LLM_Unlearning-Experiment.git'
REPO_DIR_NAME = 'PoRT_LLM_Unlearning-Experiment'
OFFICIAL_REPO = 'ChnIRuI/PoRT_LLM_Unlearning'
OPENREVIEW_ID = 'GBTUVO9vkj'
OPENREVIEW_SUPPLEMENT_URL = f'https://openreview.net/attachment?id={OPENREVIEW_ID}&name=supplementary_material'
IS_KAGGLE = Path('/kaggle/working').exists()


def has_project_layout(path):
    path = Path(path)
    return (path / 'PoRT_pipeline' / 'WMDP' / 'port_pipeline_wmdp.py').exists() and (path / 'dataset').exists()


def clone_or_use_project():
    if IS_KAGGLE:
        target = Path('/kaggle/working') / REPO_DIR_NAME
        if has_project_layout(target):
            print(f'Using existing cloned repository: {target}')
            subprocess.check_call(['git', '-C', str(target), 'pull', '--ff-only'])
            return target.resolve()
        if target.exists():
            raise RuntimeError(f'{target} exists but does not look like this repo.')
        print(f'Cloning {REPO_URL} into {target}')
        subprocess.check_call(['git', 'clone', '--depth', '1', REPO_URL, str(target)])
        return target.resolve()

    local_root = Path.cwd().resolve()
    if has_project_layout(local_root):
        return local_root
    target = local_root / REPO_DIR_NAME
    if has_project_layout(target):
        return target.resolve()
    subprocess.check_call(['git', 'clone', '--depth', '1', REPO_URL, str(target)])
    return target.resolve()


PROJECT_ROOT = clone_or_use_project()
commit_sha = subprocess.check_output(['git', '-C', str(PROJECT_ROOT), 'rev-parse', 'HEAD'], text=True).strip()
OUTPUT_DIR = Path('/kaggle/working') if IS_KAGGLE else PROJECT_ROOT / 'results'
RUN_DIR = OUTPUT_DIR / 'paper_port_official_artifact_probe'
RUN_DIR.mkdir(parents=True, exist_ok=True)

print(json.dumps({
    'project_root': str(PROJECT_ROOT),
    'commit': commit_sha,
    'run_dir': str(RUN_DIR),
    'official_repo': OFFICIAL_REPO,
    'openreview_id': OPENREVIEW_ID,
}, indent=2))


Cloning https://github.com/toanthangO20/PoRT_LLM_Unlearning-Experiment.git into /kaggle/working/PoRT_LLM_Unlearning-Experiment


Cloning into '/kaggle/working/PoRT_LLM_Unlearning-Experiment'...


{
  "project_root": "/kaggle/working/PoRT_LLM_Unlearning-Experiment",
  "commit": "6812592c3df8f763ba93da911e1a68e4e92d7e48",
  "run_dir": "/kaggle/working/paper_port_official_artifact_probe",
  "official_repo": "ChnIRuI/PoRT_LLM_Unlearning",
  "openreview_id": "GBTUVO9vkj"
}


In [2]:
def env_text(name, default=None):
    value = os.environ.get(name)
    if value is None:
        return default
    value = value.strip()
    return value if value else default


def fetch_json(url, timeout=30):
    req = urllib.request.Request(url, headers={'User-Agent': 'PoRT-artifact-probe/1.0'})
    with urllib.request.urlopen(req, timeout=timeout) as resp:
        return json.loads(resp.read().decode('utf-8'))


def fetch_bytes(url, timeout=120):
    req = urllib.request.Request(url, headers={'User-Agent': 'PoRT-artifact-probe/1.0'})
    with urllib.request.urlopen(req, timeout=timeout) as resp:
        body = resp.read()
        headers = dict(resp.headers.items())
    return body, headers


def head_url(url, timeout=30):
    req = urllib.request.Request(url, method='HEAD', headers={'User-Agent': 'PoRT-artifact-probe/1.0'})
    try:
        with urllib.request.urlopen(req, timeout=timeout) as resp:
            return {'ok': True, 'status': resp.status, 'headers': dict(resp.headers.items())}
    except Exception as exc:
        return {'ok': False, 'error': repr(exc)}


def sha256_bytes(data):
    return hashlib.sha256(data).hexdigest()


def looks_like_model_file(path):
    lower = str(path).lower()
    return lower.endswith(('.safetensors', '.bin', '.pt', '.pth', '.ckpt'))


def classify_artifact_path(path):
    lower = str(path).lower()
    kind = []
    if any(token in lower for token in ['t5', 'prefix', 'compiler', 'ast']):
        kind.append('t5_or_prefix_candidate')
    if any(token in lower for token in ['classifier', 'judge', 'judgment', 'post']):
        kind.append('classifier_candidate')
    if lower.endswith('.safetensors'):
        kind.append('safetensors')
    if lower.endswith(('.bin', '.pt', '.pth', '.ckpt')):
        kind.append('torch_checkpoint')
    return kind


def summarize_local_path(value):
    if not value:
        return {'provided': False}
    path = Path(value).expanduser()
    info = {'provided': True, 'value': value, 'exists': path.exists(), 'is_file': path.is_file(), 'is_dir': path.is_dir()}
    if path.exists() and path.is_file():
        info['size'] = path.stat().st_size
        info['suffix'] = path.suffix
    if path.exists() and path.is_dir():
        names = sorted(child.name for child in path.iterdir())[:50]
        info['top_level_files'] = names
        info['has_config_json'] = (path / 'config.json').exists()
        info['has_model_weights'] = any((path / name).exists() for name in ['model.safetensors', 'pytorch_model.bin'])
        info['has_tokenizer'] = any((path / name).exists() for name in ['tokenizer.json', 'tokenizer_config.json', 'spiece.model', 'sentencepiece.bpe.model'])
    return info


probe = {
    'project': {'project_root': str(PROJECT_ROOT), 'commit': commit_sha},
    'sources': {},
    'env_artifacts': {},
    'detected_candidates': [],
    'conclusion': {},
}
print('Helpers ready')


Helpers ready


In [3]:
github_result = {'repo': OFFICIAL_REPO}
try:
    repo_api = f'https://api.github.com/repos/{OFFICIAL_REPO}'
    repo_meta = fetch_json(repo_api)
    releases = fetch_json(f'{repo_api}/releases')
    tags = fetch_json(f'{repo_api}/tags')
    default_branch = repo_meta.get('default_branch', 'main')
    tree = fetch_json(f'{repo_api}/git/trees/{default_branch}?recursive=1')
    files = [item for item in tree.get('tree', []) if item.get('type') == 'blob']
    file_paths = [item.get('path', '') for item in files]
    model_candidates = [path for path in file_paths if looks_like_model_file(path) or classify_artifact_path(path)]
    github_result.update({
        'ok': True,
        'html_url': repo_meta.get('html_url'),
        'default_branch': default_branch,
        'releases_count': len(releases),
        'release_names': [release.get('name') or release.get('tag_name') for release in releases[:20]],
        'tags_count': len(tags),
        'tag_names': [tag.get('name') for tag in tags[:20]],
        'file_count': len(file_paths),
        'model_candidate_paths': model_candidates[:100],
    })
    for path in model_candidates:
        probe['detected_candidates'].append({'source': 'github', 'path': path, 'kind': classify_artifact_path(path)})
except Exception as exc:
    github_result.update({'ok': False, 'error': repr(exc)})

probe['sources']['github_official_repo'] = github_result
print(json.dumps(github_result, indent=2, ensure_ascii=False)[:4000])


{
  "repo": "ChnIRuI/PoRT_LLM_Unlearning",
  "ok": true,
  "html_url": "https://github.com/ChnIRuI/PoRT_LLM_Unlearning",
  "default_branch": "main",
  "releases_count": 0,
  "release_names": [],
  "tags_count": 0,
  "tag_names": [],
  "file_count": 119,
  "model_candidate_paths": [
    "dataset/AST/demonstrations.json",
    "dataset/TOFU/noise_prefix/forget01.json",
    "dataset/TOFU/noise_prefix/forget01_perturbed.json",
    "dataset/TOFU/noise_prefix/forget05.json",
    "dataset/TOFU/noise_prefix/forget05_perturbed.json",
    "dataset/TOFU/noise_prefix/forget10.json",
    "dataset/TOFU/noise_prefix/forget10_perturbed.json",
    "dataset/WMDP/noise_prefix/bio/dataset_dict.json",
    "dataset/WMDP/noise_prefix/bio/test/data-00000-of-00001.arrow",
    "dataset/WMDP/noise_prefix/bio/test/dataset_info.json",
    "dataset/WMDP/noise_prefix/bio/test/state.json",
    "dataset/WMDP/noise_prefix/chem/dataset_dict.json",
    "dataset/WMDP/noise_prefix/chem/test/data-00000-of-00001.arrow",
    "

In [4]:
supplement_result = {'url': OPENREVIEW_SUPPLEMENT_URL}
try:
    started = time.perf_counter()
    body, headers = fetch_bytes(OPENREVIEW_SUPPLEMENT_URL, timeout=180)
    elapsed = time.perf_counter() - started
    zip_path = RUN_DIR / 'openreview_supplement.zip'
    zip_path.write_bytes(body)
    entries = []
    model_candidates = []
    if zipfile.is_zipfile(BytesIO(body)):
        with zipfile.ZipFile(BytesIO(body)) as zf:
            for item in zf.infolist():
                entry = {'filename': item.filename, 'file_size': item.file_size}
                entries.append(entry)
                if looks_like_model_file(item.filename) or classify_artifact_path(item.filename):
                    model_candidates.append(entry)
    supplement_result.update({
        'ok': True,
        'download_seconds': elapsed,
        'content_length': len(body),
        'sha256': sha256_bytes(body),
        'content_type': headers.get('Content-Type') or headers.get('content-type'),
        'zip_path': str(zip_path),
        'entry_count': len(entries),
        'first_entries': entries[:50],
        'model_candidates': model_candidates[:100],
    })
    for entry in model_candidates:
        probe['detected_candidates'].append({'source': 'openreview_supplement', 'path': entry['filename'], 'size': entry['file_size'], 'kind': classify_artifact_path(entry['filename'])})
except Exception as exc:
    supplement_result.update({'ok': False, 'error': repr(exc)})

probe['sources']['openreview_supplement'] = supplement_result
print(json.dumps({k: v for k, v in supplement_result.items() if k != 'first_entries'}, indent=2, ensure_ascii=False)[:5000])


{
  "url": "https://openreview.net/attachment?id=GBTUVO9vkj&name=supplementary_material",
  "ok": true,
  "download_seconds": 0.48168611500000225,
  "content_length": 21254743,
  "sha256": "ec4f23ae73de4ea52db82921795cb41370363bc1a544650e32bb1d52347465b4",
  "content_type": "application/zip",
  "zip_path": "/kaggle/working/paper_port_official_artifact_probe/openreview_supplement.zip",
  "entry_count": 331,
  "model_candidates": [
    {
      "filename": "PoRT/.git/hooks/post-update.sample",
      "file_size": 189
    },
    {
      "filename": "PoRT/dataset/AST/",
      "file_size": 0
    },
    {
      "filename": "PoRT/dataset/AST/demonstrations.json",
      "file_size": 92167
    },
    {
      "filename": "PoRT/dataset/TOFU/noise_prefix/",
      "file_size": 0
    },
    {
      "filename": "PoRT/dataset/TOFU/noise_prefix/forget01.json",
      "file_size": 24849
    },
    {
      "filename": "PoRT/dataset/TOFU/noise_prefix/forget01_perturbed.json",
      "file_size": 102110
    },

In [5]:
hf_queries = [
    'PoRT_LLM_Unlearning',
    'Robust LLM Unlearning Post Judgment',
    'GBTUVO9vkj',
    'ChnIRuI',
    'PoRT unlearning',
    'post judgment unlearning classifier',
]
hf_result = {'queries': []}
seen_models = {}

for query in hf_queries:
    url = 'https://huggingface.co/api/models?search=' + urllib.parse.quote(query) + '&limit=20'
    query_result = {'query': query, 'url': url}
    try:
        models = fetch_json(url, timeout=30)
        compact = []
        for model in models:
            model_id = model.get('modelId')
            if not model_id:
                continue
            compact_model = {
                'modelId': model_id,
                'downloads': model.get('downloads'),
                'likes': model.get('likes'),
                'pipeline_tag': model.get('pipeline_tag'),
            }
            compact.append(compact_model)
            seen_models[model_id] = compact_model
        query_result.update({'ok': True, 'models': compact})
    except Exception as exc:
        query_result.update({'ok': False, 'error': repr(exc)})
    hf_result['queries'].append(query_result)

try:
    author_models = fetch_json('https://huggingface.co/api/models?author=ChnIRuI&limit=100', timeout=30)
    hf_result['author_ChnIRuI'] = [
        {'modelId': model.get('modelId'), 'downloads': model.get('downloads'), 'likes': model.get('likes'), 'pipeline_tag': model.get('pipeline_tag')}
        for model in author_models
    ]
    for model in hf_result['author_ChnIRuI']:
        if model.get('modelId'):
            seen_models[model['modelId']] = model
except Exception as exc:
    hf_result['author_ChnIRuI_error'] = repr(exc)

candidate_models = []
for model_id, model in sorted(seen_models.items()):
    lower = model_id.lower()
    if any(token in lower for token in ['port', 'unlearn', 'classifier', 'judgment', 't5', 'chnirui']):
        candidate_models.append(model)
        probe['detected_candidates'].append({'source': 'huggingface', 'model_id': model_id, 'kind': classify_artifact_path(model_id)})
hf_result['candidate_models'] = candidate_models
probe['sources']['huggingface_search'] = hf_result
print(json.dumps({'candidate_models': candidate_models, 'query_count': len(hf_queries)}, indent=2, ensure_ascii=False)[:5000])


{
  "candidate_models": [
    {
      "modelId": "ChnIRuI/tofu_Llama-2-7b-chat-hf_forget01_GradAscent",
      "downloads": 0,
      "likes": 0,
      "pipeline_tag": null
    }
  ],
  "query_count": 6
}


In [6]:
official_env = {
    'PORT_T5_MODEL_PATH': env_text('PORT_T5_MODEL_PATH'),
    'PORT_T5_MODEL_HF_REPO': env_text('PORT_T5_MODEL_HF_REPO'),
    'PORT_T5_MODEL_URL': env_text('PORT_T5_MODEL_URL'),
    'PORT_CLASSIFIER_BASE_MODEL': env_text('PORT_CLASSIFIER_BASE_MODEL'),
    'PORT_CLASSIFIER_HEAD_CKPT': env_text('PORT_CLASSIFIER_HEAD_CKPT'),
    'PORT_CLASSIFIER_HEAD_URL': env_text('PORT_CLASSIFIER_HEAD_URL'),
}

env_result = {'raw': official_env, 'checks': {}}

t5_path_or_repo = official_env['PORT_T5_MODEL_PATH'] or official_env['PORT_T5_MODEL_HF_REPO']
classifier_base = official_env['PORT_CLASSIFIER_BASE_MODEL']
classifier_head = official_env['PORT_CLASSIFIER_HEAD_CKPT']

if official_env['PORT_T5_MODEL_PATH']:
    env_result['checks']['t5_model_path'] = summarize_local_path(official_env['PORT_T5_MODEL_PATH'])
if official_env['PORT_T5_MODEL_HF_REPO']:
    model_id = official_env['PORT_T5_MODEL_HF_REPO']
    try:
        env_result['checks']['t5_model_hf_repo'] = {'provided': True, 'model_id': model_id, 'api': fetch_json('https://huggingface.co/api/models/' + urllib.parse.quote(model_id, safe=''), timeout=30)}
    except Exception as exc:
        env_result['checks']['t5_model_hf_repo'] = {'provided': True, 'model_id': model_id, 'ok': False, 'error': repr(exc)}
if official_env['PORT_T5_MODEL_URL']:
    env_result['checks']['t5_model_url'] = head_url(official_env['PORT_T5_MODEL_URL'])

if classifier_base:
    try:
        env_result['checks']['classifier_base_model'] = {'provided': True, 'model_id': classifier_base, 'api': fetch_json('https://huggingface.co/api/models/' + urllib.parse.quote(classifier_base, safe=''), timeout=30)}
    except Exception as exc:
        env_result['checks']['classifier_base_model'] = {'provided': True, 'model_id': classifier_base, 'ok': False, 'error': repr(exc)}

if classifier_head:
    env_result['checks']['classifier_head_ckpt'] = summarize_local_path(classifier_head)
if official_env['PORT_CLASSIFIER_HEAD_URL']:
    env_result['checks']['classifier_head_url'] = head_url(official_env['PORT_CLASSIFIER_HEAD_URL'])

missing_for_official = []
if not (official_env['PORT_T5_MODEL_PATH'] or official_env['PORT_T5_MODEL_HF_REPO'] or official_env['PORT_T5_MODEL_URL']):
    missing_for_official.append('PORT_T5_MODEL_PATH or PORT_T5_MODEL_HF_REPO or PORT_T5_MODEL_URL')
if not classifier_base:
    missing_for_official.append('PORT_CLASSIFIER_BASE_MODEL')
if not (official_env['PORT_CLASSIFIER_HEAD_CKPT'] or official_env['PORT_CLASSIFIER_HEAD_URL']):
    missing_for_official.append('PORT_CLASSIFIER_HEAD_CKPT or PORT_CLASSIFIER_HEAD_URL')

env_result['missing_for_official_mode'] = missing_for_official
env_result['official_env_complete'] = not missing_for_official
probe['env_artifacts'] = env_result
print(json.dumps(env_result, indent=2, ensure_ascii=False)[:6000])


{
  "raw": {
    "PORT_T5_MODEL_PATH": null,
    "PORT_T5_MODEL_HF_REPO": null,
    "PORT_T5_MODEL_URL": null,
    "PORT_CLASSIFIER_BASE_MODEL": null,
    "PORT_CLASSIFIER_HEAD_CKPT": null,
    "PORT_CLASSIFIER_HEAD_URL": null
  },
  "checks": {},
  "missing_for_official_mode": [
    "PORT_T5_MODEL_PATH or PORT_T5_MODEL_HF_REPO or PORT_T5_MODEL_URL",
    "PORT_CLASSIFIER_BASE_MODEL",
    "PORT_CLASSIFIER_HEAD_CKPT or PORT_CLASSIFIER_HEAD_URL"
  ],
  "official_env_complete": false
}


In [7]:
public_candidates = probe['detected_candidates']
strong_checkpoint_candidates = []
for item in public_candidates:
    path = item.get('path') or item.get('model_id') or ''
    kinds = item.get('kind') or []
    if any(kind in kinds for kind in ['safetensors', 'torch_checkpoint']) or looks_like_model_file(path):
        strong_checkpoint_candidates.append(item)

official_env_complete = probe['env_artifacts'].get('official_env_complete', False)
public_checkpoint_found = bool(strong_checkpoint_candidates)

probe['conclusion'] = {
    'official_env_complete': official_env_complete,
    'public_checkpoint_found': public_checkpoint_found,
    'strong_checkpoint_candidates': strong_checkpoint_candidates,
    'can_run_port_official_mode_now': bool(official_env_complete),
    'can_claim_paper_checkpoint_reproduction': bool(official_env_complete and public_checkpoint_found),
    'recommendation': None,
}

if official_env_complete:
    probe['conclusion']['recommendation'] = 'Run notebook 13 or a single-job PoRT smoke with PORT_ARTIFACT_MODE=official, then inspect validity and rethink metrics.'
elif public_checkpoint_found:
    probe['conclusion']['recommendation'] = 'Inspect detected public checkpoint candidates manually, map them to T5/classifier roles, then set PORT_* env vars.'
else:
    probe['conclusion']['recommendation'] = 'No public official checkpoint was found by this probe. Recreate T5/classifier artifacts from public code/data and label them as recreated, not official.'

probe_path = RUN_DIR / 'official_artifact_probe.json'
probe_path.write_text(json.dumps(probe, indent=2, ensure_ascii=False, default=str), encoding='utf-8')

summary_md = RUN_DIR / 'official_artifact_probe_summary.md'
summary_md.write_text(
    '# PoRT Official Artifact Probe Summary\n\n'
    f'- Project commit: `{commit_sha}`\n'
    f'- Official repo: `{OFFICIAL_REPO}`\n'
    f'- OpenReview ID: `{OPENREVIEW_ID}`\n'
    f'- Public checkpoint found: `{public_checkpoint_found}`\n'
    f'- Official env complete: `{official_env_complete}`\n'
    f'- Can run official mode now: `{probe["conclusion"]["can_run_port_official_mode_now"]}`\n'
    f'- Recommendation: {probe["conclusion"]["recommendation"]}\n',
    encoding='utf-8'
)

print(json.dumps(probe['conclusion'], indent=2, ensure_ascii=False))
print('Probe JSON:', probe_path)
print('Summary MD:', summary_md)


{
  "official_env_complete": false,
  "public_checkpoint_found": false,
  "strong_checkpoint_candidates": [],
  "can_run_port_official_mode_now": false,
  "can_claim_paper_checkpoint_reproduction": false,
  "recommendation": "No public official checkpoint was found by this probe. Recreate T5/classifier artifacts from public code/data and label them as recreated, not official."
}
Probe JSON: /kaggle/working/paper_port_official_artifact_probe/official_artifact_probe.json
Summary MD: /kaggle/working/paper_port_official_artifact_probe/official_artifact_probe_summary.md


## Verify Probe Outputs


In [8]:
required_files = [
    RUN_DIR / 'official_artifact_probe.json',
    RUN_DIR / 'official_artifact_probe_summary.md',
]
missing = [str(path) for path in required_files if not path.exists()]
if missing:
    raise RuntimeError(f'Missing probe outputs: {missing}')

for source_name in ['github_official_repo', 'openreview_supplement', 'huggingface_search']:
    if source_name not in probe['sources']:
        raise RuntimeError(f'Missing source audit: {source_name}')

print('PAPER PORT OFFICIAL ARTIFACT PROBE COMPLETED')
print('Can run official mode now:', probe['conclusion']['can_run_port_official_mode_now'])
print('Public checkpoint found:', probe['conclusion']['public_checkpoint_found'])
print('Recommendation:', probe['conclusion']['recommendation'])
print('Artifacts:', RUN_DIR)


PAPER PORT OFFICIAL ARTIFACT PROBE COMPLETED
Can run official mode now: False
Public checkpoint found: False
Recommendation: No public official checkpoint was found by this probe. Recreate T5/classifier artifacts from public code/data and label them as recreated, not official.
Artifacts: /kaggle/working/paper_port_official_artifact_probe
